# Milestone 3

#### This milestone is designed to familiarize you with the core mechanics of Retrieval-Augmented Generation (RAG). You will learn how to query a vector index for relevant context and use a Cross-Encoder to re-rank the results for maximum accuracy. You will then run side-by-side A/B tests to compare a model's zero-shot inference with and without this context. Finally, we will explore the physical limits of context windows (tuning the correct number of chunks to retrieve) and demonstrate the catastrophic dangers of feeding an LLM incorrect data.

### Pre-Requisite code to run !

In [ ]:
!pip install faiss-cpu #install FAISS

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating nowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


#### for q1, q2, q6

 Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points) 

In [4]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
result = zs(prompt_150, candidate_labels=labels_150)

label_to_score = dict(zip(result['labels'], result['scores']))

predicted_score = label_to_score[ans_150]

print(f"Predicted probability score for the ground-truth option: {predicted_score:.3f}")

Predicted probability score for the ground-truth option: 0.384


Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [ ]:
query_embedding = model.encode([prompt_150])

distances, indices = index.search(query_embedding, 10)

retrieved_indices = indices[0].tolist()

if 150 in retrieved_indices:
    rank = retrieved_indices.index(150) + 1
    print(f"The true correct document was placed at rank: {rank}")
else:
    print("The true correct document was not found in the top 10 retrieved documents.")

The true correct document was placed at rank: 10


Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [ ]:
wrong_context = kb[999]

adversarial_prompt = f"""Use the following context to answer the question.
Context: {wrong_context}

Question: {prompt_150}"""

adv_result = zs(adversarial_prompt, candidate_labels=labels_150)

adv_label_to_score = dict(zip(adv_result['labels'], adv_result['scores']))

adversarial_score = adv_label_to_score[ans_150]

print(f"Probability of the correct option under Adversarial RAG: {adversarial_score:.3f}")

Probability of the correct option under Adversarial RAG: 0.465


Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)
In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder. 
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context. 

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

[huggingface CrossEncoder link](https://huggingface.co/cross-encoder)
```python
#Code to use a Cross-Encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair
```
Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

reranked_results = sorted(zip(ce_scores, retrieved_indices), key=lambda x: x[0], reverse=True)

reranked_indices = [idx for score, idx in reranked_results]

if 150 in reranked_indices:
    ce_rank = reranked_indices.index(150) + 1
    print(f"The Cross-Encoder placed the true correct document at rank: {ce_rank}")
else:
    print("The true correct document is not in the reranked list.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The Cross-Encoder placed the true correct document at rank: 1


Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: \[concatenated_docs] Question: \[prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

prompt_42 = str(train.iloc[42]['prompt']) 

query_embedding_42 = model.encode([prompt_42])
distances_42, indices_42 = index.search(query_embedding_42, 5)
retrieved_indices_42 = indices_42[0].tolist()

docs_5 = [kb[i] for i in retrieved_indices_42]
concatenated_docs = " ".join(docs_5)

formatted_string = f"Context: {concatenated_docs} Question: {prompt_42}"

tokens_42 = bert_tokenizer(formatted_string, truncation=False)
total_tokens = len(tokens_42['input_ids'])

print(f"Total tokens generated: {total_tokens}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Total tokens generated: 216


Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: \[true_document] Question: \[prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [ ]:
true_doc_150 = kb[150]

rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

rag_result = zs(rag_string_150, candidate_labels=labels_150)

rag_label_to_score = dict(zip(rag_result['labels'], rag_result['scores']))

new_predicted_score = rag_label_to_score[ans_150]

print(f"New predicted probability score with true context: {new_predicted_score:.3f}")

New predicted probability score with true context: 0.989


Concept: The Danger of Bad Retrieval (Adversarial RAG) 
The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [ ]:
wrong_context = kb[999]

adversarial_string = f"Context: {wrong_context} Question: {prompt_150}"

adv_result = zs(adversarial_string, candidate_labels=labels_150)

adv_label_to_score = dict(zip(adv_result['labels'], adv_result['scores']))

adversarial_score = adv_label_to_score[ans_150]

print(f"Probability of the correct option under Adversarial RAG: {adversarial_score:.3f}")

Probability of the correct option under Adversarial RAG: 0.529


In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [ ]:
hit_count = 0
total_rows = 100

for i in range(total_rows):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    
    correct_option_letter = row['answer']
    correct_text_string = str(row[correct_option_letter])
    
    query_embedding = model.encode([prompt])
    
    distances, indices = index.search(query_embedding, 5)
    retrieved_indices = indices[0].tolist()
    retrieved_docs = [kb[idx] for idx in retrieved_indices if idx < len(kb)]
    
    is_hit = any(correct_text_string in doc for doc in retrieved_docs)
    
    if is_hit:
        hit_count += 1

hit_rate_percentage = (hit_count / total_rows) * 100

print(f"Exact Hit Rate percentage: {hit_rate_percentage:.1f}%")

Exact Hit Rate percentage: 73.0%


Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:
Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.
Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.
Augment: Create your RAG string exactly formatted as: "Context: \[best_document] Question: \[prompt]".
Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.
Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [ ]:
def calculate_map3(predictions, ground_truth):
    score = 0.0
    for rank, pred in enumerate(predictions):
        if pred == ground_truth:
            score = 1.0 / (rank + 1)
            break
    return score

map3_scores = []
total_eval_rows = 20

print("Processing evaluation loop...")

for i in range(total_eval_rows):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    ground_truth_letter = str(row['answer'])
    
    text_to_letter = {str(row[let]): let for let in ['A', 'B', 'C', 'D', 'E']}
    candidate_labels = list(text_to_letter.keys())
    
    query_embedding = model.encode([prompt])
    distances, indices = index.search(query_embedding, 5)
    retrieved_indices = indices[0].tolist()
    retrieved_docs = [kb[idx] for idx in retrieved_indices if idx < len(kb)]
    
    ce_pairs = [[prompt, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(ce_pairs)
    
    best_doc_idx = np.argmax(ce_scores)
    best_document = retrieved_docs[best_doc_idx]
    
    rag_string = f"Context: {best_document} Question: {prompt}"
    
    result = zs(rag_string, candidate_labels=candidate_labels)
    
    top_3_letters = [text_to_letter[label] for label in result['labels'][:3]]
    
    row_map3 = calculate_map3(top_3_letters, ground_truth_letter)
    map3_scores.append(row_map3)

final_average_map3 = np.mean(map3_scores)

print(f"\nFinal Average MAP@3 Score: {final_average_map3:.3f}")

Processing evaluation loop...

Final Average MAP@3 Score: 0.975
